In [1]:
import pandas as pd
import os
import shutil
import numpy as np
import subprocess
import glob
import sys
import gzip
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from io import StringIO

from google.cloud import storage

In [2]:
# Supress copy warning.
pd.options.mode.chained_assignment = None

In [3]:
# Google Cloud file access
# reads in file from google cloud folder
def blob_as_csv(bucket, path, sep='\s+', header='infer'):
    blob = bucket.get_blob(path)
    blob = blob.download_as_bytes()
    blob = str(blob, 'utf-8')
    blob = StringIO(blob)
    df = pd.read_csv(blob, sep=sep, header=header)
    return df

# gets folders from Google Cloud
def get_gcloud_bucket(bucket_name): 
    storage_client = storage.Client(project='gp2-release-terra')
    bucket = storage_client.bucket(bucket_name, user_project='gp2-release-terra')
    return bucket

In [5]:
def plot_duration_values(df, metric, metric_name, min_outcome = 0, max_outcome = 6):
    # Define bin edges
    bin_edges = np.arange(0, df["disease_duration"].max() + 3, 3)
    bin_labels = [f"{int(b)}-{int(b+3)}" for b in bin_edges[:-1]]

    # Assign bins to a new column
    df["duration_bins"] = pd.cut(df["disease_duration"], bins=bin_edges, labels=bin_labels, include_lowest=True)

    # Count occurrences per bin
    bin_counts = df["duration_bins"].value_counts()

    # Separate bins with only one value
    single_value_bins = bin_counts[bin_counts == 1].index
    multi_value_bins = bin_counts[bin_counts > 1].index

    df_single = df[df["duration_bins"].isin(single_value_bins)]
    df_multi = df[df["duration_bins"].isin(multi_value_bins)]

    # Creates violin plots - turned off outlier points
    fig = px.violin(df_multi, x="duration_bins", y=metric, box=True, points=False,
                    title=f"{metric_name} Distribution Across Disease Duration Bins",
                    category_orders={"duration_bins": bin_labels})

    # Add scatter plot for bins with only one value
    fig.add_trace(px.scatter(df_single, x="duration_bins", y=metric).data[0])

    # Update axes labels
    fig.update_xaxes(title_text="Disease Duration (Years)")
    fig.update_yaxes(title_text=metric_name)
    # fig.update_layout(yaxis=dict(range=[min_outcome, max_outcome], showgrid=True))
    fig.data[0].update(span = [min_outcome, max_outcome], spanmode='manual')

    return fig

In [6]:
def plot_duration_values(df, metric, metric_name, min_outcome=0, max_outcome=6):
    # Define bin edges with final bin as 18+
    # bin_edges = list(np.arange(0, 18, 3)) + [np.inf]
    # bin_labels = [f"{int(b)}-{int(b+3)}" for b in bin_edges[:-2]] + ["18+"]
    bin_edges = list(np.arange(0, 20, 3)) + [np.inf]
    bin_labels = [f"{int(b)}-{int(b+3)}" for b in bin_edges[:-2]] + ["20+"]

    # Assign bins to a new column
    df["duration_bins"] = pd.cut(df["disease_duration"], bins=bin_edges, labels=bin_labels, include_lowest=True)

    # Count occurrences per bin
    bin_counts = df["duration_bins"].value_counts()

    # Separate bins with only one value
    single_value_bins = bin_counts[bin_counts == 1].index
    multi_value_bins = bin_counts[bin_counts > 1].index

    df_single = df[df["duration_bins"].isin(single_value_bins)]
    df_multi = df[df["duration_bins"].isin(multi_value_bins)]

    # Creates violin plots - turned off outlier points
    fig = px.violin(df_multi, x="duration_bins", y=metric, box=True, points=False,
                    title=f"{metric_name} Distribution Across Disease Duration Bins",
                    category_orders={"duration_bins": bin_labels})

    # Add scatter plot for bins with only one value
    fig.add_trace(px.scatter(df_single, x="duration_bins", y=metric).data[0])

    # Update axes labels
    fig.update_xaxes(title_text="Disease Duration (Years)")
    fig.update_yaxes(title_text=metric_name)
    fig.data[0].update(span=[min_outcome, max_outcome], spanmode='manual')

    return fig


### Plot Data

### CS Plots

In [ ]:
# CISI-PD (one cohort)
gp2_data_bucket = get_gcloud_bucket('gp2_working_eu')

results_path = f'lietsel/Data/CISI-PD/blaacpd_cisipd_cs_2025-02-14.csv'
cisi = blob_as_csv(gp2_data_bucket, results_path, sep=',')
cisi.drop(columns = 'Unnamed: 0', inplace = True)
# display(cisi)
print(cisi.columns)

In [397]:
counts = cisi[cisi.GP2_phenotype_for_qc == 'PD']
print(len(counts.GP2ID.unique()))

105


In [293]:
# 'CISI-PD Motor Complications': 'code_cisi_pd_motor_complications'
outcomes = {'CISI-PD Motor Signs': 'code_cisi_pd_motor',
            'CISI-PD Disability': 'code_cisi_pd_disability',
            'CISI-PD Cognitive Status': 'code_cisi_pd_cognitive'}

for i in range(len(outcomes.keys())):
    outcome_key = list(outcomes.keys())[i]

    display(cisi[outcomes[outcome_key]].describe())
    fig = plot_duration_values(cisi, outcomes[outcome_key], outcome_key)
    fig.show()

count    105.000000
mean       2.885714
std        1.103441
min        0.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        5.000000
Name: code_cisi_pd_motor, dtype: float64

count    105.000000
mean       2.257143
std        1.487309
min        0.000000
25%        1.000000
50%        2.000000
75%        3.000000
max        6.000000
Name: code_cisi_pd_disability, dtype: float64

count    105.000000
mean       1.295238
std        1.358193
min        0.000000
25%        0.000000
50%        1.000000
75%        2.000000
max        5.000000
Name: code_cisi_pd_cognitive, dtype: float64

In [11]:
gp2_data_bucket = get_gcloud_bucket('gp2_working_eu')

results_path = f'lietsel/Data/MDS UPDRS/converted_mds_updrs_iii_cs_2025-04-11.csv'

pt3_cs = blob_as_csv(gp2_data_bucket, results_path, sep=',')
pt3_cs.drop(columns = 'Unnamed: 0', inplace = True)
fig = plot_duration_values(pt3_cs, 'mds_updrs_part_iii_summary_score', 'MDS-UPDRS Part III Score', max_outcome=132)
fig.show()

### LT Plots
Summary score over time and maybe showing ON versus OFF state

In [ ]:
# MDS-UPDRS PT 3 LT
# results_path = f'lietsel/Data/MDS UPDRS/mds_updrs_iii_lt_2025-04-09.csv'
results_path = f'lietsel/Data/MDS UPDRS/converted_mds_updrs_iii_lt_2025-04-11.csv'

pt3 = blob_as_csv(gp2_data_bucket, results_path, sep=',')
pt3.drop(columns = 'Unnamed: 0', inplace = True)

display(pt3.clinical_study.value_counts())
# display(pt3)

clinical_study
PPMI          12768
NET_PD_LS1     3351
Name: count, dtype: int64

In [406]:
counts = pt3[pt3.GP2_phenotype_for_qc == 'PD']
print(len(counts.GP2ID.unique()))

917


In [275]:
check = pt3[pt3.clinical_study == 'NET_PD_LS1']
# check = pt3[pt3.clinical_study == 'PPMI']
vm_count = pd.DataFrame(check.visit_month.value_counts())
vm_count.reset_index(inplace = True)
vm_count.sort_values(by = 'visit_month', inplace = True)
display(vm_count)

,visit_month,count
0,0.0,524
54,1.0,1
35,2.0,2
1,3.0,502
14,4.0,14
41,5.0,1
42,7.0,1
44,8.0,1
45,9.0,1
33,11.0,2


In [13]:
# Ensure visit_month and score are numeric
pt3['visit_month'] = pd.to_numeric(pt3['visit_month'], errors='coerce')
pt3['mds_updrs_part_iii_summary_score'] = pd.to_numeric(pt3['mds_updrs_part_iii_summary_score'], errors='coerce')

# Drop missing values
df = pt3.dropna(subset=['visit_month', 'mds_updrs_part_iii_summary_score', 'clinical_study'])

# Group by visit_month and medication state, then take mean
grouped = df.groupby(['visit_month', 'clinical_study'], as_index=False)['mds_updrs_part_iii_summary_score'].mean()

# Plot the averaged lines
fig = px.line(
    grouped,
    x='visit_month',
    y='mds_updrs_part_iii_summary_score',
    color='clinical_study',
    markers=True,
    title='Average MDS-UPDRS Part III Score Over Time by Cohort',
    labels={
        'visit_month': 'Visit Month',
        'mds_updrs_part_iii_summary_score': 'Average MDS-UPDRS Part III Score',
        'clinical_study': 'Clinical Cohort'
    }
)

fig.update_layout(
    xaxis=dict(dtick=10),
    legend_title_text='Clinical Cohort'
)

fig.show()

/opt/anaconda3/envs/python_3_9/lib/python3.9/site-packages/plotly/express/_core.py:1992: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [344]:

# Ensure visit_month and score are numeric
pt3['visit_month'] = pd.to_numeric(pt3['visit_month'], errors='coerce')
pt3['mds_updrs_part_iii_summary_score'] = pd.to_numeric(pt3['mds_updrs_part_iii_summary_score'], errors='coerce')

# Drop missing values
df = pt3.dropna(subset=['visit_month', 'mds_updrs_part_iii_summary_score', 'clinical_study'])

# Bin by every 12 months (i.e., 1 year) and label as year
df['visit_year'] = (df['visit_month'] // 12).astype(int)

# Group by binned year and cohort, then take mean
grouped = df.groupby(['visit_year', 'clinical_study'], as_index=False)['mds_updrs_part_iii_summary_score'].mean()

# Plot the averaged lines
fig = px.line(
    grouped,
    x='visit_year',
    y='mds_updrs_part_iii_summary_score',
    color='clinical_study',
    markers=True,
    title='Average MDS-UPDRS Part III Score Over Time by Cohort',
    labels={
        'visit_year': 'Visit Year',
        'mds_updrs_part_iii_summary_score': 'Average MDS-UPDRS Part III Score',
        'clinical_study': 'Clinical Cohort'
    }
)

fig.update_layout(
    xaxis=dict(dtick=1),  # tick every year
    legend_title_text='Clinical Cohort'
)

fig.show()

/opt/anaconda3/envs/python_3_9/lib/python3.9/site-packages/plotly/express/_core.py:1992: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [315]:
# Ensure visit_month and score are numeric
pt3['visit_month'] = pd.to_numeric(pt3['visit_month'], errors='coerce')
pt3['mds_updrs_part_iii_summary_score'] = pd.to_numeric(pt3['mds_updrs_part_iii_summary_score'], errors='coerce')

# Drop missing values
df = pt3.dropna(subset=['visit_month', 'mds_updrs_part_iii_summary_score', 'clinical_study'])

# Group by visit_month and medication state, then take mean
grouped = df.groupby(['visit_month', 'clinical_study'], as_index=False)['mds_updrs_part_iii_summary_score'].mean()

# Plot the averaged lines
fig = px.scatter(
    grouped,
    x='visit_month',
    y='mds_updrs_part_iii_summary_score',
    color='clinical_study',
    trendline='lowess',
    opacity=0,
    title='Average MDS-UPDRS Part III Score Over Time by Cohort',
    labels={
        'visit_month': 'Visit Month',
        'mds_updrs_part_iii_summary_score': 'Average MDS-UPDRS Part III Score',
        'clinical_study': 'Clinical Cohort'
    }
)

fig.update_layout(
    xaxis=dict(dtick=10),
    legend_title_text='Clinical Cohort'
)

fig.update_xaxes(range=[0, grouped['visit_month'].max()])

fig.show()

/opt/anaconda3/envs/python_3_9/lib/python3.9/site-packages/plotly/express/_core.py:1992: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [ ]:
# # Ensure visit_month and score are numeric
# pt3['visit_month'] = pd.to_numeric(pt3['visit_month'], errors='coerce')
# pt3['mds_updrs_part_iii_summary_score'] = pd.to_numeric(pt3['mds_updrs_part_iii_summary_score'], errors='coerce')

# # Drop missing values
# df = pt3.dropna(subset=['visit_month', 'mds_updrs_part_iii_summary_score', 'upd23b_clinical_state_on_medication'])

# # Group by visit_month and medication state, then take mean
# grouped = df.groupby(['visit_month', 'upd23b_clinical_state_on_medication'], as_index=False)['mds_updrs_part_iii_summary_score'].mean()

# # Plot the averaged lines
# fig = px.line(
#     grouped,
#     x='visit_month',
#     y='mds_updrs_part_iii_summary_score',
#     color='upd23b_clinical_state_on_medication',
#     markers=True,
#     title='Average MDS-UPDRS Part III Score Over Time by Medication State',
#     labels={
#         'visit_month': 'Visit Month',
#         'mds_updrs_part_iii_summary_score': 'Average MDS-UPDRS Part III Score',
#         'upd23b_clinical_state_on_medication': 'Medication State'
#     }
# )

# fig.update_layout(
#     xaxis=dict(dtick=10),
#     legend_title_text='Medication State'
# )

# fig.show()

### TD/PIGD Ratio

In [358]:
# PPMI Cohort
results_path = f'lietsel/Data/MDS UPDRS/ppmi_mdsupdrs_example_2025-04-09.csv'
ppmi = blob_as_csv(gp2_data_bucket, results_path, sep=',')

ppmi.drop(columns = 'Unnamed: 0', inplace = True)
ppmi.drop_duplicates(inplace = True)
print(ppmi.columns)
display(ppmi.visit_month.value_counts())
# display(ppmi)

Index(['clinical_study', 'GP2ID', 'study_arm', 'study_type',
       'GP2_phenotype_for_qc', 'visit_month', 'biological_sex_for_qc',
       'age_at_baseline', 'age_of_onset', 'age_at_diagnosis',
       'disease_duration', 'code_upd2212_walking_and_balance',
       'code_upd2213_freezing', 'mds_updrs_part_ii_summary_score',
       'code_upd2310_gait', 'code_upd2311_freezing_of_gait',
       'code_upd2312_postural_stability', 'upd23a_medication_for_pd',
       'upd23b_clinical_state_on_medication', 'code_upd2210_tremor',
       'code_upd2315a_postural_tremor_of_right_hand',
       'code_upd2315b_postural_tremor_of_left_hand',
       'code_upd2316a_kinetic_tremor_of_right_hand',
       'code_upd2316b_kinetic_tremor_of_left_hand',
       'code_upd2317a_rest_tremor_amplitude_right_upper_extremity',
       'code_upd2317b_rest_tremor_amplitude_left_upper_extremity',
       'code_upd2317c_rest_tremor_amplitude_right_lower_extremity',
       'code_upd2317d_rest_tremor_amplitude_left_lower_extrem

visit_month
 48.0     530
 36.0     518
 24.0     505
 60.0     487
 12.0     479
 72.0     437
-1.0      431
 0.0      416
 84.0     410
 6.0      397
 3.0      379
 18.0     377
 30.0     350
 42.0     336
 9.0      333
 54.0     304
 96.0     180
 108.0     25
Name: count, dtype: int64

In [359]:
td = ['code_upd2210_tremor',
    'code_upd2315a_postural_tremor_of_right_hand',
    'code_upd2315b_postural_tremor_of_left_hand',
    'code_upd2316a_kinetic_tremor_of_right_hand',
    'code_upd2316b_kinetic_tremor_of_left_hand',
    'code_upd2317a_rest_tremor_amplitude_right_upper_extremity',
    'code_upd2317b_rest_tremor_amplitude_left_upper_extremity',
    'code_upd2317c_rest_tremor_amplitude_right_lower_extremity',
    'code_upd2317d_rest_tremor_amplitude_left_lower_extremity',
    'code_upd2317e_rest_tremor_amplitude_lip_or_jaw',
    'code_upd2318_consistency_of_rest_tremor']

pigd = ['code_upd2212_walking_and_balance',
        'code_upd2213_freezing',
        'code_upd2310_gait',
        'code_upd2311_freezing_of_gait',
        'code_upd2312_postural_stability']

display(ppmi[td].isnull().any())
display(ppmi[pigd].isnull().any())

ppmi.dropna(subset=td, inplace = True)
ppmi.dropna(subset=pigd, inplace = True)

display(ppmi[td].isnull().any())
display(ppmi[pigd].isnull().any())

code_upd2210_tremor                                          True
code_upd2315a_postural_tremor_of_right_hand                  True
code_upd2315b_postural_tremor_of_left_hand                   True
code_upd2316a_kinetic_tremor_of_right_hand                   True
code_upd2316b_kinetic_tremor_of_left_hand                    True
code_upd2317a_rest_tremor_amplitude_right_upper_extremity    True
code_upd2317b_rest_tremor_amplitude_left_upper_extremity     True
code_upd2317c_rest_tremor_amplitude_right_lower_extremity    True
code_upd2317d_rest_tremor_amplitude_left_lower_extremity     True
code_upd2317e_rest_tremor_amplitude_lip_or_jaw               True
code_upd2318_consistency_of_rest_tremor                      True
dtype: bool

code_upd2212_walking_and_balance    True
code_upd2213_freezing               True
code_upd2310_gait                   True
code_upd2311_freezing_of_gait       True
code_upd2312_postural_stability     True
dtype: bool

code_upd2210_tremor                                          False
code_upd2315a_postural_tremor_of_right_hand                  False
code_upd2315b_postural_tremor_of_left_hand                   False
code_upd2316a_kinetic_tremor_of_right_hand                   False
code_upd2316b_kinetic_tremor_of_left_hand                    False
code_upd2317a_rest_tremor_amplitude_right_upper_extremity    False
code_upd2317b_rest_tremor_amplitude_left_upper_extremity     False
code_upd2317c_rest_tremor_amplitude_right_lower_extremity    False
code_upd2317d_rest_tremor_amplitude_left_lower_extremity     False
code_upd2317e_rest_tremor_amplitude_lip_or_jaw               False
code_upd2318_consistency_of_rest_tremor                      False
dtype: bool

code_upd2212_walking_and_balance    False
code_upd2213_freezing               False
code_upd2310_gait                   False
code_upd2311_freezing_of_gait       False
code_upd2312_postural_stability     False
dtype: bool

In [ ]:
baseline = ppmi[ppmi.visit_month == 0]
baseline['TD'] = baseline[td].sum(axis=1)
baseline['PIGD'] = baseline[pigd].sum(axis=1)
display(baseline)

In [ ]:
ppmi['TD'] = ppmi[td].sum(axis=1)
ppmi['PIGD'] = ppmi[pigd].sum(axis=1)

ppmi = ppmi[ppmi.PIGD != 0]

ppmi['TD_PIGD_ratio'] = ppmi['TD']/ppmi['PIGD']
display(ppmi.head())
print(len(ppmi.GP2ID.unique()))

ppmi.TD_PIGD_ratio.describe()

In [ ]:
# Ensure TD and PIGD are numeric
baseline['TD'] = pd.to_numeric(baseline['TD'], errors='coerce')
baseline['PIGD'] = pd.to_numeric(baseline['PIGD'], errors='coerce')

# Filter out missing values
df = baseline[['TD', 'PIGD']].dropna()
df.drop_duplicates(inplace = True)

# Define x range for the reference lines
x_min = df['TD'].min()
x_max = df['TD'].max()
x_vals = [x_min, x_max]

# Create the scatter plot
fig = go.Figure()

# Add scatter points - DOUBLE CHECK THIS INFO TOMORROW
fig.add_trace(go.Scatter(
    x=df['TD'],
    y=df['PIGD'],
    mode='markers',
    name='Data Points',
    marker=dict(color='blue')
))

# Add reference line with slope = 0.9
fig.add_trace(go.Scatter(
    x=x_vals,
    y=[(1/0.9) * x for x in x_vals],
    mode='lines',
    name='Slope = 1/0.9',
    line=dict(dash='dash', color='red')
))

# Add reference line with slope = 1.15
fig.add_trace(go.Scatter(
    x=x_vals,
    y=[(1/1.15) * x for x in x_vals],
    mode='lines',
    name='Slope = 1/1.15',
    line=dict(dash='dot', color='green')
))

# Customize layout
fig.update_layout(
    # title='TD vs PIGD with Reference Slopes',
    title='PIGD vs TD with Reference Slopes',
    xaxis_title='TD',
    yaxis_title='PIGD',
    legend_title='Legend'
)

fig.show()

In [ ]:
# Ensure visit_month and score are numeric
ppmi['visit_month'] = pd.to_numeric(ppmi['visit_month'], errors='coerce')

# Drop missing values
df = ppmi[['GP2ID', 'visit_month', 'TD_PIGD_ratio']].dropna()
df.drop_duplicates(inplace = True)

display(df)

# Plot each year's extent data in a stacked line chart:
fig = px.line(df, 
              x='visit_month', 
              y='TD_PIGD_ratio', 
              line_group='GP2ID', 
              color='GP2ID',
              labels={'visit_month': 'Visit Month', 'TD_PIGD_ratio': 'TD/PIGD Ratio'},
              title='TD/PIGD Ratio Over Time Per Sample')

# Draw border around the plot:
fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True)
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True)

fig.show()

In [216]:
ppmi[pigd].isnull().any()

code_upd2212_walking_and_balance    False
code_upd2213_freezing               False
code_upd2310_gait                   False
code_upd2311_freezing_of_gait       False
code_upd2312_postural_stability     False
dtype: bool

In [206]:
def avg_plot(check, title):

    # Ensure visit_month and score are numeric
    check['visit_month'] = pd.to_numeric(check['visit_month'], errors='coerce')

    # Drop missing values
    df = check.dropna(subset=['visit_month', 'TD_PIGD_ratio'])

    # Group by visit_month and medication state, then take mean
    grouped = df.groupby(['visit_month'], as_index=False)['TD_PIGD_ratio'].mean()
    display(grouped)

    # Plot the averaged lines
    fig = px.line(
        grouped,
        x='visit_month',
        y='TD_PIGD_ratio',
        markers=True,
        title=f'{title} Average TD/PIGD Ratio Over Time',
        labels={
            'visit_month': 'Visit Month',
            'TD_PIGD_ratio': 'Average TD/PIGD Ratio'
        }
    )

    fig.show()

In [207]:
def spaghetti_plot(check, title):
    # Ensure visit_month and score are numeric
    check['visit_month'] = pd.to_numeric(check['visit_month'], errors='coerce')

    # Drop missing values
    df = check[['GP2ID', 'visit_month', 'TD_PIGD_ratio']].dropna()
    df.drop_duplicates(inplace = True)

    # Plot each year's extent data in a stacked line chart:
    fig = px.line(df, 
                x='visit_month', 
                y='TD_PIGD_ratio', 
                line_group='GP2ID', 
                color='GP2ID',
                labels={'visit_month': 'Visit Month', 'TD_PIGD_ratio': 'TD/PIGD Ratio'},
                title=f'{title} TD/PIGD Ratio Over Time Per Sample')

    # Draw border around the plot:
    fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True)
    fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True)

    fig.show()

In [482]:
id_td = ppmi[(ppmi.TD_PIGD_ratio>=1.15)].GP2ID.unique()
id_pigd = ppmi[(ppmi.TD_PIGD_ratio<=0.9)].GP2ID.unique()
id_td_stable = np.setdiff1d(id_td, id_pigd) # --> Plot 1
plot1 = ppmi[ppmi.GP2ID.isin(id_td_stable)]
print(len(plot1.GP2ID.unique()))
print(len(id_td_stable))
# spaghetti_plot(plot1, 'TD Stable')
avg_plot(plot1, 'TD Stable')

id_pigd_stable = np.setdiff1d(id_pigd, id_td) # --> Plot 2
plot2 = ppmi[ppmi.GP2ID.isin(id_pigd_stable)]
print(len(plot2.GP2ID.unique()))
print(len(id_pigd_stable)) 
# spaghetti_plot(plot2, 'PIGD Stable')
avg_plot(plot2, 'PIGD Stable')

id_unstable = np.intersect1d(id_td, id_pigd) # --> Plot 3
plot3 = ppmi[ppmi.GP2ID.isin(id_unstable)]
print(len(plot3.GP2ID.unique()))
print(len(id_unstable))
# spaghetti_plot(plot3, 'Unstable')
avg_plot(plot3, 'Unstable Phenotype Group')

id_other = np.setdiff1d(ppmi.GP2ID.unique(), np.union1d(id_td, id_pigd)) # --> Plot 4
plot4 = ppmi[ppmi.GP2ID.isin(id_other)]
print(len(plot4.GP2ID.unique()))
print(len(id_other))
# spaghetti_plot(plot4, 'Other')
avg_plot(plot4, 'Other')


215
215


,visit_month,TD_PIGD_ratio
0,-1.0,5.354321
1,0.0,5.431871
2,3.0,6.130342
3,6.0,6.721429
4,9.0,5.847470
5,12.0,6.811247
6,18.0,6.470964
7,24.0,6.623472
8,30.0,5.824541
9,36.0,6.262619


21
21


,visit_month,TD_PIGD_ratio
0,-1.0,0.213542
1,0.0,0.363636
2,3.0,0.217949
3,6.0,0.156410
4,9.0,0.188462
5,12.0,0.173077
6,18.0,0.296296
7,24.0,0.222222
8,30.0,0.305556
9,36.0,0.136508


183
183


,visit_month,TD_PIGD_ratio
0,-1.0,2.879055
1,0.0,2.885195
2,3.0,2.811695
3,6.0,3.125042
4,9.0,3.015019
5,12.0,2.712863
6,18.0,2.598190
7,24.0,2.549939
8,30.0,2.139824
9,36.0,2.250123


1
1


,visit_month,TD_PIGD_ratio
0,-1.0,1.0


In [289]:
plot3_vm = pd.DataFrame(plot3.visit_month.value_counts())
plot3_vm.reset_index(inplace = True)
plot3_vm.sort_values(by = 'visit_month', inplace = True)
display(plot3_vm)

,visit_month,count
11,-1.0,138
9,0.0,142
4,3.0,153
6,6.0,146
13,9.0,131
2,12.0,154
1,18.0,155
0,24.0,156
8,30.0,145
5,36.0,152


In [480]:
### Jittered Scatter Plot 1st visit vs. Last
def prep_df(ppmi_tmp):
    last_visit = ppmi_tmp.groupby(['GP2ID'], as_index=False)['visit_month'].max()
    first_visit = ppmi_tmp.groupby(['GP2ID'], as_index=False)['visit_month'].min()
    # display(last_visit)
    # display(first_visit)

    last_visit.rename(columns = {'visit_month': 'last_visit'}, inplace = True)
    first_visit.rename(columns = {'visit_month': 'first_visit'}, inplace = True)

    final_info = first_visit.merge(last_visit, on = 'GP2ID', how = 'inner')

    # Step 2: Merge to get score at first and last visits
    # First visit scores
    score_first = ppmi_tmp.merge(first_visit, left_on=['GP2ID', 'visit_month'], right_on=['GP2ID', 'first_visit'], how='inner')
    score_first = score_first[['GP2ID', 'TD_PIGD_ratio']].rename(columns={'TD_PIGD_ratio': 'score_first'})

    # Last visit scores
    score_last = ppmi_tmp.merge(last_visit, left_on=['GP2ID', 'visit_month'], right_on=['GP2ID', 'last_visit'], how='inner')
    score_last = score_last[['GP2ID', 'TD_PIGD_ratio']].rename(columns={'TD_PIGD_ratio': 'score_last'})

    # Step 3: Merge all together
    final_info = final_info.merge(score_first, on='GP2ID', how='left')
    final_info = final_info.merge(score_last, on='GP2ID', how='left')

    return final_info

def add_jitter(arr, scale=0.1):
    return arr + np.random.normal(scale=scale, size=len(arr))

def plot_interactive_first_vs_last(df_sf):

    # Add jitter to scores
    df_sf['First Score (Jittered)'] = add_jitter(df_sf['score_first'])
    df_sf['Last Score (Jittered)'] = add_jitter(df_sf['score_last'])

    # Create the plot
    fig = px.scatter(
        df_sf, 
        x='First Score (Jittered)', 
        y='Last Score (Jittered)', 
        hover_data=['GP2ID'],
        title='Comparison of First vs Last TD/PIGD Ratio per Sample'
    )

    # Render the plot in Streamlit
    fig.show()

def plot_interactive_first_vs_last_strata(df_sf, strata):

    # Add jitter to scores
    df_sf['First Score (Jittered)'] = add_jitter(df_sf['score_first'])
    df_sf['Last Score (Jittered)'] = add_jitter(df_sf['score_last'])

    # Define custom color map
    df_sf[strata] = df_sf[strata].map({
        1: 'TD Stable',
        2: 'PIGD Stable',
        3: 'TD → PIGD',
        4: 'PIGD → TD'
    })

    color_map = {
        'TD Stable': '#1f78b4',   # dark teal
        'PIGD Stable': '#1b9e77', # dark green (color-blind safe)
        'TD → PIGD': '#7570b3',   # dark purple
        'PIGD → TD': '#e7298a'    # dark magenta
    }

    # Create the plot
    fig = px.scatter(
        df_sf, 
        x='First Score (Jittered)', 
        y='Last Score (Jittered)', 
        color = strata,
        color_discrete_map=color_map,
        hover_data=['GP2ID', 'score_first', 'score_last'],
        title='Comparison of First vs Last TD/PIGD Ratio per Sample'
    )

    fig.update_layout(
        legend_title_text='Phenotype Group'
    )

    # Render the plot in Streamlit
    return fig

In [ ]:
ppmi_tmp = ppmi.copy()
prepped = prep_df(ppmi_tmp)

# no strata b/c PD
plot_interactive_first_vs_last(prepped)

In [ ]:
df = prepped.copy()

# Assuming your DataFrame is called df
conditions = [
    (df['score_first'] >= 1.15) & (df['score_last'] >= 1.15),  # TD stable
    (df['score_first'] <= 0.9) & (df['score_last'] <= 0.9),    # PIGD stable
    (df['score_first'] >= 1.15) & (df['score_last'] <= 0.9),   # TD to PIGD
    (df['score_first'] <= 0.9) & (df['score_last'] >= 1.15)    # PIGD to TD
]

values = [1, 2, 3, 4]

df['group'] = np.select(conditions, values, default=np.nan)  # default is NaN if no condition matches
print(len(df[df.group.isnull()]))
display(df)

fig = plot_interactive_first_vs_last_strata(df, 'group')
fig.write_html("PD_subtypes_TD_PIGD_ratio.html")
fig.show()

In [477]:
visit_df = df[df.last_visit >= 50]

fig = plot_interactive_first_vs_last_strata(visit_df, 'group')
fig.show()

/opt/anaconda3/envs/python_3_9/lib/python3.9/site-packages/plotly/express/_core.py:1992: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [395]:
id_td = ppmi[(ppmi.TD_PIGD_ratio>=1.15)].GP2ID.unique()
id_pigd = ppmi[(ppmi.TD_PIGD_ratio<=0.9)].GP2ID.unique()
id_td_stable = np.setdiff1d(id_td, id_pigd) # --> Plot 1
plot1 = ppmi[ppmi.GP2ID.isin(id_td_stable)]
print(len(plot1.GP2ID.unique()))
print(len(id_td_stable))
plot1_prepped = prep_df(plot1)
plot_interactive_first_vs_last(plot1_prepped)

id_pigd_stable = np.setdiff1d(id_pigd, id_td) # --> Plot 2
plot2 = ppmi[ppmi.GP2ID.isin(id_pigd_stable)]
print(len(plot2.GP2ID.unique()))
print(len(id_pigd_stable)) 
plot2_prepped = prep_df(plot2)
plot_interactive_first_vs_last(plot2_prepped)

id_unstable = np.intersect1d(id_td, id_pigd) # --> Plot 3
plot3 = ppmi[ppmi.GP2ID.isin(id_unstable)]
print(len(plot3.GP2ID.unique()))
print(len(id_unstable))
plot3_prepped = prep_df(plot3)
plot_interactive_first_vs_last(plot3_prepped)

id_other = np.setdiff1d(ppmi.GP2ID.unique(), np.union1d(id_td, id_pigd)) # --> Plot 4
plot4 = ppmi[ppmi.GP2ID.isin(id_other)]
print(len(plot4.GP2ID.unique()))
print(len(id_other))
plot4_prepped = prep_df(plot4)
plot_interactive_first_vs_last(plot4_prepped)


215
215


21
21


183
183


1
1
